# Silver Layer – Dublin Bikes

Read the Bronze Delta table, clean, standardize, and enrich the data to create a trusted Silver dataset for downstream analytics.

In [0]:
from pyspark.sql.functions import (
    col,
    current_timestamp,
    sum,
    when
)

In [0]:
df = spark.read.table("urban_mobility.bronze.stations_raw")

total_records = df.count()

display(df.limit(5))

print(f"Bronze records: {total_records}")

In [0]:
duplicate_df = (
    df.groupBy("station_id", "last_reported")
      .count()
      .filter(col("count") > 1)
)

duplicate_count = duplicate_df.count()

display(duplicate_df)

print(f"Duplicate groups: {duplicate_count}")

In [0]:
# Count invalid records
invalid_capacity_count = invalid_capacity.count()

# Create the Silver DataFrame
silver_df = (
    df.filter(
        (col("num_bikes_available") + col("num_docks_available")) <= col("capacity")
    )
    .withColumnRenamed("lat", "latitude")
    .withColumnRenamed("lon", "longitude")
    .withColumn("updated_timestamp", current_timestamp())
)

# Count output records
silver_records = silver_df.count()

print(f"Total records: {total_records}")
print(f"Invalid capacity records: {invalid_capacity_count}")
print(f"Silver records: {silver_records}")

In [0]:
null_counts = silver_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in silver_df.columns
])

display(null_counts)

In [0]:
invalid_values = silver_df.filter(
    (col("num_bikes_available") < 0) |
    (col("num_docks_available") < 0) |
    (col("capacity") < 0)
)

invalid_value_count = invalid_values.count()

display(invalid_values)

print(f"Invalid numeric records: {invalid_value_count}")

In [0]:
silver_df.groupBy(
    "is_installed",
    "is_renting",
    "is_returning"
).count().orderBy("count", ascending=False).show(truncate=False)

In [0]:
future_timestamp_count = silver_df.filter(
    col("last_reported") > current_timestamp()
).count()

print(f"Future timestamp records: {future_timestamp_count}")

In [0]:
invalid_coordinate_count = silver_df.filter(
    (col("latitude") < -90) | (col("latitude") > 90) |
    (col("longitude") < -180) | (col("longitude") > 180)
).count()

print(f"Invalid coordinate records: {invalid_coordinate_count}")

In [0]:
silver_records = silver_df.count()

print(f"Silver records: {silver_records}")

In [0]:
(
    silver_df
      .write
      .format("delta")
      .mode("overwrite")
      .saveAsTable("urban_mobility.silver.stations")
)